### E.3 Lab 2 — CHSH Correlation Sweep

### Lab Access and Execution Guide

This guide explains how to run and explore the hands-on quantum computing labs that accompany the book  
**Quantum AI Systems: Theory, Architecture, and Applications** (Professional and Student Volumes).  

The labs are an integral part of the MyQuantumBook project, designed to reinforce key concepts from the chapters through interactive exploration. They are built for execution on **Google Colab** and **IBM Quantum backends** using **Qiskit**, and follow the IEEE-compliant figure, caption, and documentation standards described in the text.  

Each lab is cross-referenced to its corresponding chapter and appendix figure (Appendix E), ensuring reproducibility and scholarly traceability. 

**Getting Started**
1. Launch the notebook in Google Colab using the provided badge.
2. Run the setup cells to install Qiskit:
   `!pip install qiskit`

**Using IBM Quantum Systems**
1. Sign up at https://quantum.ibm.com and create an API token.
2. Run the IBMQ setup cell.
3. Replace 'MY_API_TOKEN' with your real token (only needed once).
4. Select backends using `provider.get_backend('ibmq_qasm_simulator')` or others.

**Lab Structure**
Each code section aligns with a chapter from the book.
- Modify and re-run code blocks.
- View circuits with `.draw()`.
- Apply to custom inputs to deepen your understanding.

**Additional Help**
- Refer to the Qiskit Documentation: https://qiskit.org/documentation/
- For support, contact your course instructor or visit the IBM Quantum Community forums.

3. Or launch this lab directly now: [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jopaneur/QuantumAI-Labs/blob/main/Advanced_Labs/notebooks/Chapter_5_Correlation_Entanglement_and_Nonlocality_Advanced_Challenge_CHSH_Correlation_Sweep.ipynb)


---



**Note for Lab Participants**
Each plot generate in this notebook is automatically saved as a `.png` file under: Advanced_Labs/figures/

The filenames follow the Appendix E figure numbering (e.g., `E2_1_Bloch_Trajectories.png`, `E2_6_DensityMatrix_Heatmap.png`).  

This allows you to both view results inline in Colab **and** find the corresponding image files for reports, submissions, or cross-references in the book.

**Where Figures Are Saved**
- In **Google Colab**, the images are created inside the session’s working directory at:  
  `/content/Advanced_Labs/figures/`  

- When running **locally**, they appear next to your notebook files, under the subfolder:  
  `Advanced_Labs/figures/`  

- These images are **not automatically added to your GitHub repo**. They will only appear there if you manually copy, commit, and push them.

**Customizing Save Location**
If you want the figures saved elsewhere, you can change the `subdir` default in the `save_e_figure()` helper or pass a different path each time you call it.


---

**Chapter 1 — Foundations of Quantum AI Systems**

Chapter 1 lays the groundwork for understanding how quantum resources extend beyond classical information models. It introduces entanglement as a nonclassical correlation mechanism and highlights Bell-style inequalities as the empirical test that separates quantum predictions from classical hidden-variable theories. The CHSH framework, with its classical bound of 2 and quantum limit of 2√2 ≈ 2.828, is presented as both a scientific milestone and a system-level benchmark for QAIS.

Lab 2 brings this foundation to life. By constructing entangled states, varying measurement settings, and computing the CHSH statistic, learners see how quantum correlations manifest directly in data. The exercise reinforces Chapter 1’s theme: logical nonlocality is not an abstraction, but a measurable property that defines the frontier of quantum AI systems.

**Advanced Lab 2 - CHSH Correlation Sweep**

**Goal:** Implement and analyze a CHSH (Clauser–Horne–Shimony–Holt) correlation sweep to test Bell inequality violations. This lab prepares entangled states, varies measurement settings, and computes correlations across angles to reveal quantum versus classical bounds. By comparing observed CHSH values to the classical limit of 2, learners confirm the distinct nonlocal correlations enabled by entanglement. This hands-on exercise connects the chapter’s theme of logical nonlocality to measurable outcomes, reinforcing why quantum systems outperform classical hidden-variable models. Cross-reference: Appendix E.2, Figures E.2.2a–e.

**Task 1 - Helper Utilities**

In [ ]:
# ---- Figure helper (robust; standardized for all labs) ----
import os, matplotlib.pyplot as plt

def save_e_figure(fig_label: str,
                  fname: str,
                  subdir: str = "Advanced_Labs/figures",
                  fig=None, ax=None):
    """
    Save the current/explicit figure with a consistent IEEE-compliant path.
    - Does NOT inject figure numbers/titles into the visual.
    - Appends .png if missing.
    - Uses bbox_inches='tight' to avoid clipping.
    """
    os.makedirs(subdir, exist_ok=True)

    if fig is None:
        fig = plt.gcf()
    if ax is None:
        ax = fig.axes[0] if fig.axes else None
    if ax is None:
        print("⚠️ No axes found. Draw a plot first, or pass fig/ax explicitly.")
        return

    # Keep plot titles free of figure numbers; captions carry the E.2.x label.

    # Ensure .png extension
    if not fname.lower().endswith(".png"):
        fname += ".png"

    outpath = os.path.join(subdir, fname)
    fig.tight_layout()
    fig.savefig(outpath, dpi=160, bbox_inches="tight")

    print(f"Saved figure → {outpath}\n{fig_label}")


**Methodology Analysis**

Methodology Analysis
This helper standardizes figure saving for Appendix E. It creates the destination folder if needed, enforces a .png extension, applies tight_layout() plus bbox_inches="tight" to prevent clipping, and does not inject figure numbers into plot titles (IEEE style keeps numbers in captions). The function accepts either the current active figure or explicit figure/axis handles, then emits a confirmation line with the full path and the figure label for traceability.

**Participant Feedback**

Participant Feedback
Running this cell alone will not draw a plot. Later, when you generate a figure and call save_e_figure(...), the console should print a confirmation like:

* Saved figure → Advanced_Labs/figures/P2_AdvLab02_E.2.2a.png
Figure E.2.2a


* Use filenames that match the figure numbers (e.g., P2_AdvLab02_E.2.2a.png, P2_AdvLab02_E.2.2b.png) so Appendix E cross-references remain consistent.

**Task 2 - Environment Setup and Package Validation**

In [ ]:
# === Environment Setup (CPU-only, idempotent) — Advanced Lab 2: CHSH Correlation Sweep ===
# Safe to re-run. Installs missing packages quietly, then prints versions.
import sys, subprocess, importlib.util

def ensure(pip_name, module_name=None):
    """
    Install `pip_name` if the importable `module_name` is missing.
    If module_name is None, it defaults to pip_name with '-' replaced by '_'.
    """
    mod = module_name or pip_name.replace("-", "_")
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

# Core scientific + quantum stack (CPU simulators only)
ensure("numpy")
ensure("matplotlib")
ensure("qiskit")
ensure("qiskit-aer", "qiskit_aer")

# Imports after ensuring installation
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import qiskit
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Operator, SparsePauliOp
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

# Reproducibility + plotting defaults (useful with 5 figures)
np.random.seed(7)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 160,
    "figure.autolayout": True,
})

# CPU simulator with fixed seed for reproducible CHSH sweeps
backend = AerSimulator(seed_simulator=777)

# Version + environment report
print("Python:", sys.version.split()[0])
print("Qiskit:", qiskit.__version__)
try:
    import qiskit_aer
    print("Qiskit Aer:", qiskit_aer.__version__)
except Exception:
    print("Qiskit Aer: import failed")
print("NumPy:", np.__version__)
print("Matplotlib:", plt.matplotlib.__version__)
print("CPU Simulator:", backend.name())


**Methodology Analysis**

This cell standardizes the CPU-only environment for the CHSH correlation sweep. It installs any missing packages idempotently, then imports the stack used to: (1) build and transpile two-qubit Bell circuits, (2) compute statevectors/expectations, and (3) render five publication-quality plots consistently across runs. A fixed simulator seed AerSimulator(seed_simulator=777) ensures the CHSH angle sweeps and histogram samples are reproducible. Plot defaults and a global RNG seed stabilize figure appearance and layout so the saved outputs map cleanly to Appendix E.

**Participant Feedback**

Running the environment setup cell does not create a figure. You should see a short version report and:

* CPU Simulator: aer_simulator

If you do not see the simulator name, re-run this cell. Subsequent cells that generate the 5 plots (E.2.2a–E.2.2e) will use this backend; figures should save with the standardized helper and IEEE-style filenames (e.g., P2_AdvLab02_E.2.2a.png, …, P2_AdvLab02_E.2.2e.png).

**Lab Overview – CHSH Correlation Sweep and Quantum Nonlocality**

This lab explores one of the most profound demonstrations of quantum mechanics: the CHSH (Clauser–Horne–Shimony–Holt) inequality. Participants investigate how entangled two-qubit states produce correlations that exceed classical limits, quantifying the violation through expectation values at varying measurement angles. By sweeping through analyzer orientations, learners visualize the smooth quantum–classical transition and directly compute the Bell parameter S, which signals nonlocal behavior.

**Challenge:**

Simulate measurement correlations for a maximally entangled Bell state while varying detector settings to calculate the CHSH quantity
S = E(A,B) − E(A,B′) + E(A′,B) + E(A′,B′).
Demonstrate how quantum predictions violate the classical bound (|S| ≤ 2), and interpret this violation as evidence of entanglement and nonlocal correlations.

**Implementation Note:**

The lab uses Qiskit’s Statevector formalism to compute exact probabilities for measurement outcomes along user-defined axes, avoiding statistical sampling noise. Each correlation term is obtained through expectation values of tensor-product observables, allowing smooth angular sweeps across the measurement space. Results are visualized as both individual expectation curves and composite CHSH violations. This deterministic simulation provides a clean, noise-free baseline for understanding real-device deviations in later chapters.

**Expected Results**

* Figure E.1.2a (Measurement Correlations vs Angle):
Correlation values follow a cosine-like dependence on the relative analyzer angle, confirming the sinusoidal structure of quantum predictions.

* Figure E.1.2b (CHSH Parameter Sweep):
The computed S value peaks near 2√2 ≈ 2.828, violating the classical bound of 2. This maximum occurs near canonical measurement settings (0°, 45°, 22.5°, 67.5°).

* Figure E.1.2c (Classical vs Quantum Comparison):
The classical correlation curve remains confined within |S| ≤ 2, while the quantum curve visibly exceeds it, illustrating the distinction between local realism and quantum entanglement.

Together, these results highlight how quantum correlations defy classical constraints, demonstrating that no local hidden-variable model can fully explain the observed outcomes. The lab establishes the foundational intuition for later discussions of quantum communication, teleportation, and entanglement-based AI inference.

---

**Task 3 - The following cell prepares the simulator and libraries for the CHSH study.**

**Goal:** Construct a Bell state and sweep measurement angles to compute a CHSH S-value, demonstrating violation beyond classical bounds.
*Outcomes:* derive CHSH settings, implement correlator estimation, contrast ideal vs shot-based, interpret bounds (2, 2√2).

**Build CHSH Test Circuits and Estimate *S* paramter**
*What the code does*

Builds the entangled Bell state ∣Φ⁺⟩ = (∣00⟩ + ∣11⟩)/√2, applies the four CHSH measurement settings (A,B), (A,B′), (A′,B), (A′,B′), and computes correlators. A result S > 2 confirms violation of the classical bound. 

These expectation values are then combined into the CHSH statistic:
S = E(a,b) + E(a,b′) + E(a′,b) − E(a′,b′).
The result is printed as a single numerical test of Bell inequality violation.

In [ ]:
# Build CHSH test circuits and estimate S parameter




**Methodology Analysis (Single-S Computation)**

This procedure encodes two qubits into the ∣Φ⁺⟩ state and rotates each qubit into measurement bases defined by the four CHSH settings. From the sampled outcomes, correlators E are computed as differences between even- and odd-parity counts. These values are then combined into the CHSH statistic S = E(a,b) + E(a,b′) + E(a′,b) − E(a′,b′). This workflow demonstrates how to translate abstract Bell settings into concrete quantum circuits and measurement statistics.


**Participant Feedback**

The output will print the four correlators and the computed CHSH S-value.

* Expected result: S > 2, confirming violation of the classical/local-realist bound.

* Ideal case: Noiseless simulations should approach S ≈ 2.828 (2√2).

If S ≤ 2: Double-check the measurement basis rotations and bitstring mappings.

---
**Shot-Based Variant: A Hardware-Like View**

To explore a more realistic setup, you can run the following shot-based version of the CHSH sweep. In this approach, the simulator estimates correlators by sampling measurement outcomes in rotated bases, mimicking what happens on actual quantum hardware. The overall curve retains the same shape as the statevector result, but each point now carries a bit of statistical noise. Running with at least 4096 shots ensures the plot remains smooth and reliable.

**Task 4 - CHSH Correlation Sweep (Shot-based)**

*What the code does: Sweeps Bob’s angle b, computes S(b) from finite shots, plots curve with red and green lines.*

When running the shot-based version with 4096 shots, the CHSH curve will look nearly identical to the statevector result: it will rise above the classical bound (2), peak close to 2.8, and then symmetrically decline. The key difference is that the points will not lie perfectly on the curve. Instead, you’ll see slight fluctuations around the ideal shape, reflecting statistical noise from finite sampling. If you reduce the number of shots, the noise becomes more visible; if you increase shots, the curve smooths out and approaches the exact statevector plot.

In [ ]:
# === CHSH Correlation Sweep — Shot-based (Qiskit Aer) ===
# Builds |Φ+> = (|00> + |11>)/√2, measures along rotated bases in the x–z plane,
# estimates correlators from samples, and plots the CHSH S-value vs Bob's angle b.



**Figure E.2.2a — CHSH Correlation Sweep (Shot-based).**

Measured CHSH S-values for |Φ+⟩ as Bob’s angle b varies. Sampled estimates rise above the classical bound S=2 and approach the Tsirelson limit 2√2 at near-optimal angles, validating nonlocal correlations with finite-shot statistics.

**Expected Results**

The S(b) curve should rise above the classical bound of 2 and peak close to 2.8 at b ≈ π/4. Shot noise introduces scatter, but the violation remains visible across a broad range of angles.

**Technical Analysis (for the visual)**

Sampling fluctuations scale as 1/√shots. With 4096 shots, the scatter is modest. The curve should be symmetric about b = 0, with a maximum near b ≈ π/4. Persistent offset below the ideal prediction suggests basis rotation or bit-order issues.

**Intuition Sidebar**

Think of entangled qubits as “quantum coins” tossed by Alice and Bob. Classically, their correlation score caps at 2. With entanglement, the score can reach 2.8, visibly crossing the classical limit in the plot.


---

**Task 5 - CHSH Statevector vs Shot-based (Side-by-side)**

*What the code does: Left panel: ideal S(b) with statevector. Right panel: sampled S(b).*

The left panel (statevector) should display a smooth, symmetric curve that rises cleanly above 2 and approaches 2.828, the Tsirelson bound.
The right panel (shot-based) should show the same shape, but with finite-shot fluctuations. Participants should be able to compare the two panels directly, seeing how the ideal theory and hardware-like sampling align, despite noise.

In [ ]:
# === CHSH Correlation Sweep: Statevector vs Shot-based Comparison ===
# Compares ideal statevector evaluation with hardware-like sampling using finite shots.




**Figure E.2.2b — CHSH Statevector vs Shot-based.**

The ideal (left) sets the analytic benchmark; the sampled (right) tracks the trend with finite-shot jitter, still exceeding the classical limit.

**Expected Results**

The left panel should show a smooth curve peaking at 2√2 ≈ 2.828. The right panel should track this shape but with scatter due to finite shots. Both surpass the classical threshold of 2.

**Technical Analysis (for the visual)**

The side-by-side view demonstrates how theory and experiment align. The sampled curve should follow the ideal trend. Large systematic deviations indicate mis-specified measurement angles or transpilation mismatches.

**Intuition Sidebar**

This comparison is like showing a perfect curve (theory) beside a hand-drawn sketch (experiment). Despite the shakiness, the essential feature is preserved: both beat the classical bound.



---

**Task 6 - CHSH Correlation Sweep (Statevector, ideal)**

*What the code does: Ideal S(b) with no sampling noise.*

Participants should observe a clean, continuous curve of S(b) that smoothly crosses above 2 and reaches the quantum maximum of 2√2 ≈ 2.828 at near-optimal angles. The result should be symmetric around b = 0, showing how the cosine structure of the correlator dictates the sweep. Unlike the shot-based version, this ideal visualization contains no scatter or noise, providing the analytic benchmark for the experiment.

In [ ]:
# === CHSH Correlation Sweep: Statevector (Ideal, Noiseless) ===



**Figure E.2.2c — CHSH Correlation Sweep (Statevector, ideal)**

The noiseless curve reaches the quantum maximum 2√2 at near-optimal angles and sits cleanly above the classical bound of 2.

**Expected Results**

Students should see a perfectly smooth curve rising above 2 and peaking at 2√2 ≈ 2.828. The symmetry across b = 0 reflects the cosine form of the correlator.

**Technical Spotlight**

For the Bell state in the x–z plane, the correlator is E(a,b) = cos(a − b). With the canonical angles a = 0, a′ = π/2, b′ = −π/4, sweeping b produces the maximum violation. This analytic structure explains the clean symmetry of the statevector curve.

**Intuition Sidebar**

In the idealized case, quantum coins always cooperate at maximum strength. The perfect curve illustrates the full, noise-free advantage of entanglement over classical correlations.


---
**Task 7 - Shot-based Only**

*What this code does: Isolated plot of shot-based S(b).*

Participants should see a scattered curve of S(b) values produced from finite-shot measurements. The fluctuations will hover around the smooth quantum prediction but with small random deviations. Crucially, several points should rise above the red line at 2, confirming a Bell violation even under realistic sampling. With higher shot counts, the scatter will shrink and the curve will more closely resemble the ideal.

In [ ]:
# === CHSH Correlation Sweep: Shot-based (Finite Shots) ===




**Figure E.2.2d — CHSH Shot-based Sweep (4096 shots)**

Sampling noise produces scatter, yet the curve still surpasses the classical bound across a range of b, confirming nonlocality in a hardware-like regime.

**Expected Results**

The scatter plot should still show violations above 2 across many values of b, though each point fluctuates around the smooth ideal curve.

**Technical Analysis (for the visual)**

Shot noise causes scatter proportional to 1/√N, where N is the number of shots. Error bars or bootstrapping can quantify this. Increasing the shot count would smooth the curve and bring it closer to the ideal statevector result.

**Intuition Sidebar**

This is like rolling a die a limited number of times—you won’t get the perfect average every time, but the overall pattern emerges with more trials. Even with scatter, the quantum advantage remains visible.


---


**Task 8 - Integrated Comparison (side-by-side)**

*What the code does: Puts ideal and shot-based results together in one comparison.*

In the integrated comparison, the ideal curve should trace a smooth arc that peaks at ≈2.828, while the shot-based points should cluster around it with some spread. Participants should clearly observe the band between 2 and 2√2 framing both results. The shot-based data should never collapse to the classical threshold, demonstrating that violations remain visible even with hardware-like noise.

In [ ]:
# === CHSH Correlation Sweep: Integrated Statevector vs Shot-based ===



**Figure E.2.2e — Integrated CHSH Comparison (Ideal vs Shot-based)**

*What the code does: The ideal ceiling and the sampled curve are shown together: both exceed 2, with finite-shot fluctuations around the quantum trend.*

**Expected Results**

The ideal curve should sweep cleanly to 2√2, while the sampled data hovers near it with some spread. The key observation is that both stay above the classical line at 2.

**Technical Analysis (for the visual)**

The red line at 2 and green line at 2√2 frame the violation band. Both ideal and sampled results sit within this band, showing the robustness of nonlocal correlations. Noise introduces fluctuation, but not enough to erase the violation.

**Intuition Sidebar**

Think of the red and green lines as a “quantum performance window.” The classical world can’t reach beyond 2, but entanglement lives confidently inside the higher band. Even with noise, the results land in the quantum-only zone.



---
**Conclusion**

Across all five visuals, you verified that an entangled Bell state violates the CHSH inequality. The statevector curves provide a precise theoretical ceiling (Tsirelson bound), while the shot-based curves demonstrate that even with finite statistics, the violation persists and is experimentally observable. The agreement in shape and location of the maximum between ideal and sampled results validates your measurement design and angle conventions. Any substantial, consistent shortfall from the ideal trend would indicate configuration or basis issues rather than fundamental physics—your plots show the expected quantum behavior.

* Entangled systems violate CHSH, distinguishing quantum from classical correlations.

* Ideal vs shot-based views: theory provides the target; sampling demonstrates experimental viability.

* Any consistent shortfall from ideal suggests basis/sign/transpile issues rather than physics.

**Key Take-Aways**

CHSH sets a crisp classical boundary; quantum surpasses it.
Angles matter: correct rotations place the peak near b ≈ π/4.
Sampling vs bias: randomness narrows with shots; systematic offsets imply configuration issues.
The visual gap above 2 is the proof of nonlocality.

**Congratulations**

Outstanding work. You reproduced a canonical quantum information result with a professional workflow: state preparation → basis control → shot-based estimation → analytic benchmarking → integrated comparison. This lab bridges foundational quantum nonlocality and practical experimental realities—exactly the kind of reasoning you’ll leverage in more advanced QAIS labs where validation, uncertainty, and benchmarking determine whether a pipeline is merely elegant or genuinely sound.

### Appendix E → Appendix B Cross-Reference
See **Appendix B — Quick Self-Check, Chapter 1 — Foundations of Quantum AI Systems**:  
- Questions 6 and 7 (Bell inequality and CHSH correlation).  
They analyze the non-locality violation measured in **E.3 Lab 2** (Figures E.3.2a–e).


---
**How to save or submit your work**

- **If you are a student (graded/evaluated):**  
  1. Export your key plots or the entire notebook to PDF (File → Print/Save as PDF).  
  2. Save the notebook (`.ipynb`).  
  3. Bundle any extra files (CSVs/images) if used.  
  4. Upload to your LMS or repository as instructed (include your name and lab number).  
  5. Repro checklist: set a random seed where applicable, note backend and shots, and list package versions.  

- **If you are a professional/self‑learner (non‑graded exercise):**  
  1. Save the notebook (`File → Download .ipynb`) to your computer for personal reference.  
  2. Optionally export to PDF for archiving.  
  3. Keep any generated plots or data locally.  
  4. Use version control (GitHub, GitLab) if you wish to track your personal progress.



___